## <b>Movie Recommendation System</b>

### <b>Step 3 : Content-Based Recommendation Engine</b>

#### <b>Objective:</b>
The objective of this phase is to build a content-based recommendation system capable of suggesting movies and TV series that are similar to a selected title.

#### <b>Methodology:</b>
A TF-IDF (Term Frequency-Inverse Document Frequency) vectorizer was applied to the engineered tags feature to convert textual movie metadata into numerical feature vectors.

The input feature consisted of a combined textual representation containing:

- Genre
- Director
- Writer
- Actors
- Plot

After vectorization, cosine similarity was used to measure the similarity between every pair of movies.

#### <b>Recommendation Process</b>

1. Convert movie tags into TF-IDF vectors.
2. Compute pairwise cosine similarity.
3. Identify movies with the highest similarity scores.
4. Rank recommendations using a weighted combination of:
   - Content Similarity (80%)
   - IMDb Rating (20%)
   
#### <b>Enhanced Ranking</b>

To improve recommendation quality, IMDb ratings were incorporated into the ranking process.

Final Recommendation Score:

`Final Score = 0.8 × Similarity Score + 0.2 × Normalized IMDb Rating`

This ensures that highly similar and highly rated content is ranked above similar but poorly rated content.


#### Step 1 : Import the Libraries

In [1]:
# 1. Import the libraries
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

#### Step 2 : Load the Dataset

In [2]:
# 2. Load the dataset
movies = pd.read_csv('../data/Netflix_Dataset.csv')
print(movies.shape)

movies.head()

(3516, 4)


,imdbID,Title,imdbRating,tags
0,tt0107362,Last Action Hero,6.5,action adventure comedy johnmctiernan zakpenn ...
1,tt21191806,Back in Action,5.9,action comedy sethgordon sethgordon brendano'b...
2,tt0318155,Looney Tunes Back in Action,5.8,animation adventure comedy joedante larrydoyle...
3,tt15600222,An Action Hero,7.0,action comedy crime anirudhiyer anirudhiyer ne...
4,tt0120633,A Civil Action,6.6,biography drama stevenzaillian jonathanharr st...


#### Step 3 : Check Missing Values

In [3]:
# 3. Check Missing Values
movies.isnull().sum()

imdbID        0
Title         0
imdbRating    0
tags          0
dtype: int64

#### Step 4 : Create TF-IDF Vectorizer

In [4]:
# 4. Create a TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=10000, stop_words='english')

##### Transfer Tags into Numerical Vectors

In [5]:
tfidf_matrix = tfidf.fit_transform(movies['tags'])

In [6]:
print(tfidf_matrix.shape)

(3516, 10000)


#### Step 5 : Calculate Cosine Similarity

In [7]:
# 5. Calculate Cosine Similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [8]:
print(cosine_sim.shape)

(3516, 3516)


Every movie has similarity score with every other movie.

#### Step 6 : Create Movie Index Mapping

In [9]:
# 6. Create Movie Index Mapping
movie_indices = pd.Series(
    movies.index,
    index=movies["Title"]
)

print("Unique Titles:", movie_indices.index.nunique())
print("Total Rows:", len(movie_indices))

Unique Titles: 3317
Total Rows: 3516


In [10]:
movie_indices.head()

Title
Last Action Hero               0
Back in Action                 1
Looney Tunes Back in Action    2
An Action Hero                 3
A Civil Action                 4
dtype: int64

#### Step 7 : Build Recommendation Function

In [11]:
# 7. Build the Recommendation Function
def recommend(movie_title, top_n=10):

    matches = movies[
        movies["Title"].str.lower() == movie_title.lower()
    ]

    if len(matches) == 0:
        return "Movie not found"

    idx = matches.index[0]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    movie_indices_list = [
        i[0]
        for i in similarity_scores
    ]

    return movies.iloc[
        movie_indices_list
    ][["Title", "imdbRating"]]

In [12]:
recommend('Batman')

,Title,imdbRating
1252,Batman Returns,7.1
1071,Batman Mask of the Phantasm,7.8
1091,The Batman,7.4
1041,Batman Beyond,8.1
1534,Batman Forever,5.5
1138,Batman Beyond,6.4
1434,Batman Black and White,7.6
1499,Batman Rise of Sin Tzu,7.3
1314,The New Adventures of Batman,6.7
1117,Batman Beyond Return of the Joker,7.8


Till now recommendations are based only on content similarity.

We'll improve them by giving some importance to IMDb ratings.
Normalize Rating

#### Step 8 : Improve Ranking Using IMDb Rating

In [13]:
# 8. Add IMDb Rating Weightage
movies['rating_norm'] = (movies['imdbRating'] - movies['imdbRating'].min()) / (movies['imdbRating'].max() - movies['imdbRating'].min())

In [14]:
movies[['Title', 'imdbRating', 'rating_norm']].head()

,Title,imdbRating,rating_norm
0,Last Action Hero,6.5,0.611111
1,Back in Action,5.9,0.544444
2,Looney Tunes Back in Action,5.8,0.533333
3,An Action Hero,7.0,0.666667
4,A Civil Action,6.6,0.622222


##### Enhanced Recommendation Function

In [15]:
# 9. Create Enhanced Recommendation Functiom
def recommend_v2(movie_title, top_n=10):
    matches = movies[movies['Title'].str.lower() == movie_title.lower()]

    if len(matches) == 0:
        return "Movie not Found."
    
    idx = matches.index[0]

    similarity_scores = list(enumerate(cosine_sim[idx]))

    recommendations = []

    for movie_idx, similarity in similarity_scores:
        rating_score = movies.iloc[movie_idx]['rating_norm']

        final_score = (0.8 * similarity + 0.2 * rating_score)

        recommendations.append(
            (movie_idx, final_score, similarity)
        )
    
    recommendations = sorted(
        recommendations,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = recommendations[1:top_n+1]

    movie_ids = [i[0] for i in recommendations]

    result = movies.iloc[movie_ids][["Title", "imdbRating"]].copy()

    return result


In [16]:
recommend_v2("Ms. Marvel")

,Title,imdbRating
987,Final Fantasy Anthology,8.4
1023,Batman Beyond The Movie,7.7
635,Life of Crime 1984-2020,8.4
882,Sherina's Adventure,7.8
1580,The Hero Love Story of a Spy,5.5
997,Final Fantasy Crystal Chronicles,7.2
1344,Son of Batman,6.7
3121,Journals of Supernatural Adventure,6.6
1059,Marvel Zombies,7.0
3285,Temüjin A Supernatural Adventure,7.0


In [17]:
# Evaluate Recommendation Quality
def show_movie(title):

    result = movies[
        movies["Title"].str.lower() == title.lower()
    ]

    return result[
        ["Title", "imdbRating"]
    ]

In [18]:
show_movie("Inception")

,Title,imdbRating
37,Inception,8.6


#### Step 9 : Save the models

In [19]:
# 10. Save TF-IDF Model
import pickle
import os

os.makedirs("../models", exist_ok=True)

with open("../models/tfidf.pkl", 'wb') as file:
    pickle.dump(tfidf, file)

In [20]:
# 11. Save Similarity Matrix
with open("../models/cosine_similarity.pkl", 'wb') as file:
    pickle.dump(cosine_sim, file)

In [21]:
# 12. Save Movie Dataset
movies.to_pickle("../models/movies.pkl")

In [22]:
# 13. Verify Files
import os

print(os.listdir('../models'))

['cosine_similarity.pkl', 'movies.pkl', 'tfidf.pkl']


In [23]:
recommend_v2("Action")

,Title,imdbRating
1855,The Hollywood Knights,6.2
2133,Moguls & Movie Stars A History of Hollywood,8.5
1269,The Animation Show 2005,7.2
63,Traveling the Stars Action Bronson and Friends...,7.8
2064,The Roosevelts An Intimate History,8.8
1909,Hollywood,9.3
2723,Office Space,7.6
634,Law & Order True Crime,7.4
96,2020 Oscar Nominated Short Films Live Action,7.7
835,Dragon Ball Mystical Adventure,6.8


#### Outcomes
The model can recommend movies with similar themes, genres, actors, directors, and storylines, providing a personalized Netflix-style recommendation experience.